In [5]:
import pandas as pd

# 1. Verify Step 2 Monthly Dataset
monthly_df = pd.read_csv("D:/3rd_year/AIML/FA implementation/Nigdi_OpenAQ_Monthly_Model_Ready.csv")
print("=== MONTHLY DATASET ===")
print("Total Months:", len(monthly_df))
print("Columns:", monthly_df.columns.tolist())
print(monthly_df[["month", "pm25_mean", "pm25_max", "pm25_exceedance_days"]].head())

# 2. Verify Air Quality Resilience Scoreboard
scores_df = pd.read_csv("D:/3rd_year/AIML/FA implementation/nigdi_aq_resilience_scoreboard.csv")
print("\n=== RESILIENCE SCOREBOARD ===")
print("Total Observations:", len(scores_df))
print("Average Predicted Score:", scores_df["predicted_resilience_score"].mean().round(1), "/ 100")
print("\nAction Distribution:")
print(scores_df["resilience_category"].value_counts())


=== MONTHLY DATASET ===
Total Months: 20
Columns: ['month', 'pm25_mean', 'pm25_max', 'pm25_min', 'pm25_std', 'pm10_mean', 'pm10_max', 'no2_mean', 'o3_mean', 'wind_speed_mean', 'pm25_exceedance_days', 'pm25_severe_days', 'zone_id', 'latitude', 'longitude']
        month  pm25_mean  pm25_max  pm25_exceedance_days
0  2025-02-28      39.69     51.36                     0
1  2025-03-31      38.61     59.26                     0
2  2025-04-30      26.83     45.88                     0
3  2025-05-31      17.12     26.42                     0
4  2025-06-30      12.59     24.06                     0

=== RESILIENCE SCOREBOARD ===
Total Observations: 26
Average Predicted Score: 77.8 / 100

Action Distribution:
resilience_category
High Resilience (Clean Air)           18
Moderate Resilience (Satisfactory)     6
Vulnerable (Warning Level)             2
Name: count, dtype: int64


Step 5: Air Quality Resilience Scoreboard & Intervention Engine

This script converts the hourly PM2.5 machine learning predictions into:
1. A normalized Urban Climate Resilience Air Quality Sub-Score (0 to 100).
2. Risk classification aligned with Indian National Ambient Air Quality Standards (CPCB).
3. Actionable, explainable climate adaptation recommendations triggered by vulnerability levels.
4. Exports the complete evaluation and scoreboard dataset to 'nigdi_aq_resilience_scoreboard.csv'.

In [7]:
import os
import joblib
import pandas as pd
import numpy as np

In [8]:
def compute_aq_resilience_score(pm25_val):
    """
    Computes a 0-100 Resilience Score from PM2.5 concentration (ug/m3)
    Based on India CPCB / NAAQS thresholds:
      - NAAQS 24-hr Standard = 60 ug/m3 (Yields Score 50: Threshold of Acceptable Resilience)
      - Good Air Quality <= 30 ug/m3 (Yields Score 75 - 100: High Resilience)
      - Poor Air Quality 60 - 90 ug/m3 (Yields Score 25 - 50: Vulnerable)
      - Severe Air Quality > 90 ug/m3 (Yields Score 0 - 25: Critical Vulnerability)
    """
    if pd.isna(pm25_val):
        return np.nan
    # Linear piece-wise resilience formula
    score = 100.0 - (pm25_val / 60.0) * 50.0
    return float(np.clip(score, 0.0, 100.0))
#---------------------------------------------------------------------------------------------------------------------------------------

def get_resilience_category(score):
    if pd.isna(score):
        return "Unknown"
    elif score >= 75:
        return "High Resilience (Clean Air)"
    elif score >= 50:
        return "Moderate Resilience (Satisfactory)"
    elif score >= 25:
        return "Vulnerable (Warning Level)"
    else:
        return "Critical Vulnerability (Emergency Action)"
#---------------------------------------------------------------------------------------------------------------------------------------

def get_adaptation_recommendation(score):
    if pd.isna(score):
        return "Insufficient data for recommendation."
    elif score >= 75:
        return "Normal operations; standard green-space & urban canopy preservation."
    elif score >= 50:
        return "Traffic advisory: Optimize signal timing; incentivize public transit usage."
    elif score >= 25:
        return "Intervention: Deploy mechanical sweepers & water misting on arterial roads; restrict heavy diesel idling."
    else:
        return "Emergency Intervention: Halt active construction dust emissions; enforce low-emission zone restrictions; issue sensitive-group health advisory."
#---------------------------------------------------------------------------------------------------------------------------------------

def run_aq_scoreboard():
    print("AIR QUALITY RESILIENCE SCOREBOARD & INTERVENTION ENGINE")
    print("\n\n")

    hourly_file = "Nigdi_OpenAQ_Hourly_Model_Ready.csv"
    model_file = "nigdi_next_hour_pm25_gradient_boosting.pkl"

    if not os.path.exists(hourly_file):
        raise FileNotFoundError(f"Missing {hourly_file}. Please run data prep first.")
    
    print(f"Loading hourly observations from: {hourly_file}")
    df = pd.read_csv(hourly_file)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)

    # Recreate the exact feature engineering used in AQ_score_hourly.ipynb
    df["hour"] = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek
    df["month"] = df["timestamp"].dt.month
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["wind_direction_sin"] = np.sin(np.radians(df["wind_direction"]))
    df["wind_direction_cos"] = np.cos(np.radians(df["wind_direction"]))

    for lag in [1, 2, 3, 6, 12, 24]:
        df[f"pm25_lag_{lag}"] = df["pm25"].shift(lag)

    df["pm25_mean_3"] = df["pm25"].shift(1).rolling(3).mean()
    df["pm25_mean_6"] = df["pm25"].shift(1).rolling(6).mean()
    df["pm25_mean_24"] = df["pm25"].shift(1).rolling(24).mean()

    # Target: next hour PM2.5
    df["pm25_next_hour"] = df["pm25"].shift(-1)
    df["next_timestamp"] = df["timestamp"].shift(-1)
    is_next_hour = (df["next_timestamp"] - df["timestamp"]).dt.total_seconds().eq(3600)
    df.loc[~is_next_hour, "pm25_next_hour"] = np.nan

    features = [
        "pm25", "pm10", "no2", "o3",
        "wind_speed",
        "wind_direction_sin", "wind_direction_cos",
        "hour_sin", "hour_cos",
        "day_of_week", "month",
        "pm25_lag_1", "pm25_lag_2", "pm25_lag_3",
        "pm25_lag_6", "pm25_lag_12", "pm25_lag_24",
        "pm25_mean_3", "pm25_mean_6", "pm25_mean_24"
    ]

    model_df = df[features + ["pm25_next_hour", "timestamp"]].dropna(subset=["pm25_next_hour"]).copy()
    split_index = int(len(model_df) * 0.80)
    test_df = model_df.iloc[split_index:].copy()

    # Load trained Gradient Boosting pipeline
    if os.path.exists(model_file):
        print(f"Loading trained Gradient Boosting model from: {model_file}")
        model = joblib.load(model_file)
        predictions = model.predict(test_df[features])
    else:
        print("Model file not found; fitting pipeline directly on train set...")
        from sklearn.pipeline import Pipeline
        from sklearn.impute import SimpleImputer
        from sklearn.ensemble import GradientBoostingRegressor

        train_df = model_df.iloc[:split_index]
        model = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("gradient_boosting", GradientBoostingRegressor(
                n_estimators=300, learning_rate=0.03, max_depth=3, loss="huber", random_state=42
            ))
        ])
        model.fit(train_df[features], train_df["pm25_next_hour"])
        joblib.dump(model, model_file)
        predictions = model.predict(test_df[features])

    # Build Scoreboard DataFrame
    scoreboard = pd.DataFrame({
        "timestamp": test_df["timestamp"].values,
        "zone_id": "PCMC_Nigdi",
        "actual_pm25": test_df["pm25_next_hour"].values.round(2),
        "predicted_pm25": np.round(predictions, 2)
    })

    # Compute 0-100 Resilience Scores
    scoreboard["observed_resilience_score"] = scoreboard["actual_pm25"].apply(compute_aq_resilience_score).round(1)
    scoreboard["predicted_resilience_score"] = scoreboard["predicted_pm25"].apply(compute_aq_resilience_score).round(1)

    # Classify Risk Category & Trigger Adaptation Action
    scoreboard["resilience_category"] = scoreboard["predicted_resilience_score"].apply(get_resilience_category)
    scoreboard["recommended_action"] = scoreboard["predicted_resilience_score"].apply(get_adaptation_recommendation)

    # Export Scoreboard CSV
    output_csv = "nigdi_aq_resilience_scoreboard.csv"
    scoreboard.to_csv(output_csv, index=False)

    print(f"\n\nSUCCESS: Air Quality Scoreboard exported to '{output_csv}'")
    print(f"\n\nTotal Test Observations: {len(scoreboard)}")
    print(f"\n\nAverage Predicted PM2.5: {scoreboard['predicted_pm25'].mean():.2f} ug/m3")
    print(f"\n\nAverage AQ Resilience Score: {scoreboard['predicted_resilience_score'].mean():.1f} / 100")
    print("\n\nResilience Category Breakdown:")
    print(scoreboard["resilience_category"].value_counts(normalize=True).mul(100).round(1).astype(str) + "%")

    print("\n\nScoreboard Sample (First 5 hours):")
    print(scoreboard[["timestamp", "actual_pm25", "predicted_pm25", "predicted_resilience_score", "resilience_category"]].head())

    return scoreboard

if __name__ == "__main__":
    run_aq_scoreboard()


AIR QUALITY RESILIENCE SCOREBOARD & INTERVENTION ENGINE



Loading hourly observations from: Nigdi_OpenAQ_Hourly_Model_Ready.csv
Loading trained Gradient Boosting model from: nigdi_next_hour_pm25_gradient_boosting.pkl


SUCCESS: Air Quality Scoreboard exported to 'nigdi_aq_resilience_scoreboard.csv'


Total Test Observations: 2138


Average Predicted PM2.5: 14.45 ug/m3


Average AQ Resilience Score: 88.0 / 100


Resilience Category Breakdown:
resilience_category
High Resilience (Clean Air)           98.9%
Moderate Resilience (Satisfactory)     1.1%
Name: proportion, dtype: object


Scoreboard Sample (First 5 hours):
            timestamp  actual_pm25  predicted_pm25  \
0 2026-05-16 07:30:00         6.35           11.09   
1 2026-05-16 08:30:00         4.31            8.15   
2 2026-05-16 09:30:00         2.64            7.09   
3 2026-05-16 10:30:00         8.17            9.39   
4 2026-05-16 11:30:00        22.76           12.72   

   predicted_resilience_score          resilience_c

In [1]:
import sys
print(sys.executable)

c:\Users\ACER\AppData\Local\Python\pythoncore-3.14-64\python.exe


In [2]:
!{sys.executable} C:\Users\ACER\step5_aq_resilience_scoreboard.py

STEP 5: AIR QUALITY RESILIENCE SCOREBOARD & INTERVENTION ENGINE
Loading hourly observations from: Nigdi_OpenAQ_Hourly_Model_Ready.csv
Loading trained Gradient Boosting model from: nigdi_next_hour_pm25_gradient_boosting.pkl

SUCCESS: Air Quality Scoreboard exported to 'nigdi_aq_resilience_scoreboard.csv'
Total Test Observations: 2138
Average Predicted PM2.5: 14.45 ug/m3
Average AQ Resilience Score: 88.0 / 100

Resilience Category Breakdown:
resilience_category
High Resilience (Clean Air)           98.9%
Moderate Resilience (Satisfactory)     1.1%
Name: proportion, dtype: object

Scoreboard Sample (First 5 hours):
            timestamp  ...          resilience_category
0 2026-05-16 07:30:00  ...  High Resilience (Clean Air)
1 2026-05-16 08:30:00  ...  High Resilience (Clean Air)
2 2026-05-16 09:30:00  ...  High Resilience (Clean Air)
3 2026-05-16 10:30:00  ...  High Resilience (Clean Air)
4 2026-05-16 11:30:00  ...  High Resilience (Clean Air)

[5 rows x 5 columns]


In [3]:
import pandas as pd

df = pd.read_csv("Nigdi_MultiHazard_Monthly_Composite_Scoreboard.csv")
print("Total Months Aligned:", len(df))
print("Missing Values Check:\n", df[["composite_climate_resilience_score", "aq_resilience_subscore", "heat_resilience_subscore"]].isnull().sum())
print("\nScore Ranges:")
print(df[["composite_climate_resilience_score", "aq_resilience_subscore", "heat_resilience_subscore"]].describe().round(1))


Total Months Aligned: 20
Missing Values Check:
 composite_climate_resilience_score    0
aq_resilience_subscore                0
heat_resilience_subscore              0
dtype: int64

Score Ranges:
       composite_climate_resilience_score  aq_resilience_subscore  \
count                                20.0                    20.0   
mean                                 69.0                    75.4   
std                                  14.4                    15.2   
min                                  43.1                    43.0   
25%                                  57.9                    67.6   
50%                                  68.4                    81.2   
75%                                  84.1                    86.9   
max                                  86.8                    92.1   

       heat_resilience_subscore  
count                      20.0  
mean                       77.5  
std                        27.9  
min                        29.4  
25%         

In [4]:
from IPython.display import IFrame

IFrame(src="nigdi_climate_vulnerability_map.html", width="100%", height=550)

## Remaining implementation fragments

The following cells are designed to be run in order inside Jupyter. They do not use terminal Python commands. The future-risk cell is a transparent seasonal baseline forecast for planning; it should not be described as a long-term climate projection without additional climate-model data.

In [5]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "Nigdi_TriHazard_Monthly_Master_Scoreboard.csv").exists():
    PROJECT_ROOT = Path(r"D:/3rd_year/AIML/FA implementation")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python kernel: {sys.executable}")
print(f"Pandas version: {pd.__version__}")

Project root: C:\Users\ACER
Python kernel: C:\Users\ACER\anaconda3\python.exe
Pandas version: 2.2.3


In [6]:
master_path = PROJECT_ROOT / "Nigdi_TriHazard_Monthly_Master_Scoreboard.csv"
flood_path = PROJECT_ROOT / "Nigdi_Flood_GSW_Monthly.csv"

master = pd.read_csv(master_path)
flood_source = pd.read_csv(flood_path)

required_columns = {
    "year_month",
    "aq_resilience_subscore",
    "heat_resilience_subscore",
    "flood_resilience_subscore",
    "composite_climate_resilience_score",
}
missing_columns = sorted(required_columns.difference(master.columns))
expected_composite = (
    0.40 * master["aq_resilience_subscore"]
    + 0.35 * master["heat_resilience_subscore"]
    + 0.25 * master["flood_resilience_subscore"]
).round(1)

comparison = master[["year_month", "flood_resilience_subscore"]].merge(
    flood_source[["year_month", "flood_resilience_subscore"]],
    on="year_month",
    how="left",
    suffixes=("_master", "_source"),
)

print(f"Master rows: {len(master)}")
print(f"Required columns missing: {missing_columns}")
print(f"Composite formula matches: {expected_composite.eq(master['composite_climate_resilience_score']).all()}")
print(f"Flood scores matching current source: {comparison['flood_resilience_subscore_master'].eq(comparison['flood_resilience_subscore_source']).sum()}/{len(comparison)}")
comparison.loc[~comparison['flood_resilience_subscore_master'].eq(comparison['flood_resilience_subscore_source'])].head()

Master rows: 20
Required columns missing: []
Composite formula matches: True
Flood scores matching current source: 0/20


,year_month,flood_resilience_subscore_master,flood_resilience_subscore_source
0,2025-02,55.2,90.8
1,2025-03,58.6,90.8
2,2025-04,59.0,87.6
3,2025-05,55.0,80.2
4,2025-06,58.0,43.4


In [7]:
aq = pd.read_csv(PROJECT_ROOT / "Nigdi_OpenAQ_Monthly_Model_Ready.csv")
heat = pd.read_csv(PROJECT_ROOT / "Nigdi_CDS_Heatwave_Monthly.csv")
flood = pd.read_csv(PROJECT_ROOT / "Nigdi_Flood_GSW_Monthly.csv")

# Normalize each source to the same monthly join key.
aq["year_month"] = pd.to_datetime(aq["month"], errors="coerce", dayfirst=True).dt.strftime("%Y-%m")
heat["year_month"] = pd.to_datetime(heat["month"], errors="coerce", dayfirst=True).dt.strftime("%Y-%m")
if "year_month" not in flood.columns:
    flood["year_month"] = pd.to_datetime(flood["month"], errors="coerce", dayfirst=True).dt.strftime("%Y-%m")

join_keys = ["year_month", "zone_id", "latitude", "longitude"]
aq = aq.drop(columns=["month"], errors="ignore")
heat = heat.drop(columns=["month"], errors="ignore")
flood = flood.drop(columns=["month"], errors="ignore")

fused = aq.merge(heat, on=join_keys, how="inner", suffixes=("_aq", "_heat"))
fused = fused.merge(flood, on=join_keys, how="inner", suffixes=("", "_flood"))
fused = fused.sort_values("year_month").reset_index(drop=True)

fused["aq_resilience_subscore"] = np.clip(
    100.0 - (fused["pm25_mean"] / 60.0) * 50.0, 0.0, 100.0
).round(1)
fused["heat_resilience_subscore"] = np.clip(
    100.0 - ((fused["temp_c_max"] - 30.0) / 15.0) * 100.0, 0.0, 100.0
).round(1)
fused["composite_climate_resilience_score"] = (
    0.40 * fused["aq_resilience_subscore"]
    + 0.35 * fused["heat_resilience_subscore"]
    + 0.25 * fused["flood_resilience_subscore"]
).round(1)

def classify_resilience(score):
    if score >= 75:
        return "High Resilience (Clean, Cool & Safe)"
    if score >= 50:
        return "Moderate Resilience (Satisfactory)"
    if score >= 25:
        return "Vulnerable (Elevated Hazard)"
    return "Critical Vulnerability (Emergency Action)"

def recommend_intervention(row):
    vulnerable = {
        "Air Pollution": row["aq_resilience_subscore"] < 50,
        "Heatwave": row["heat_resilience_subscore"] < 50,
        "Waterlogging/Flood": row["flood_resilience_subscore"] < 50,
    }
    active = [name for name, is_active in vulnerable.items() if is_active]
    if len(active) >= 2:
        return "COMPOUND CRISIS ALERT: " + ", ".join(active) + ". Activate multi-agency municipal disaster protocol."
    if active:
        return f"{active[0].upper()} ALERT: Activate the corresponding municipal adaptation protocol."
    return "NORMAL OPERATIONS: Continue canopy, drainage, and continuous sensor monitoring."

fused["resilience_category"] = fused["composite_climate_resilience_score"].apply(classify_resilience)
fused["recommended_urban_intervention"] = fused.apply(recommend_intervention, axis=1)

lead_columns = [
    "year_month", "zone_id", "composite_climate_resilience_score",
    "aq_resilience_subscore", "heat_resilience_subscore",
    "flood_resilience_subscore", "resilience_category",
    "recommended_urban_intervention",
]
fused = fused[lead_columns + [c for c in fused.columns if c not in lead_columns]]

# These are the exact filenames used by the Streamlit app and map generator.
fused.to_csv(PROJECT_ROOT / "Nigdi_TriHazard_Monthly_Master_Scoreboard.csv", index=False)
fused.to_csv(PROJECT_ROOT / "Nigdi_MultiHazard_Monthly_Composite_Scoreboard.csv", index=False)

print(f"Corrected fused rows: {len(fused)}")
print(f"AQ/heat/flood integration complete: {len(fused) == 20}")
print(fused[["year_month", "aq_resilience_subscore", "heat_resilience_subscore", "flood_resilience_subscore", "composite_climate_resilience_score"]].tail())

Corrected fused rows: 20
AQ/heat/flood integration complete: True
   year_month  aq_resilience_subscore  heat_resilience_subscore  \
15    2026-05                    83.5                      29.4   
16    2026-06                    88.2                      61.3   
17    2026-07                    89.3                     100.0   
18    2026-08                    86.0                     100.0   
19    2026-09                    86.5                     100.0   

    flood_resilience_subscore  composite_climate_resilience_score  
15                       80.2                                63.7  
16                       43.4                                67.6  
17                        0.0                                70.7  
18                        5.4                                70.8  
19                       34.4                                78.2  


C:\Users\ACER\AppData\Local\Temp\ipykernel_24584\2736017823.py:7: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  heat["year_month"] = pd.to_datetime(heat["month"], errors="coerce", dayfirst=True).dt.strftime("%Y-%m")


In [8]:
# Transparent six-month seasonal baseline for planning support.
# This uses the latest available value for each calendar month; it is not a climate-change projection.
forecast_source = fused.copy() if "fused" in globals() else master.copy()
forecast_source["date"] = pd.to_datetime(forecast_source["year_month"] + "-01")
forecast_source = forecast_source.sort_values("date")
last_date = forecast_source["date"].max()
future_dates = pd.date_range(last_date + pd.offsets.MonthBegin(1), periods=6, freq="MS")

score_columns = [
    "aq_resilience_subscore",
    "heat_resilience_subscore",
    "flood_resilience_subscore",
]
forecast_rows = []
for future_date in future_dates:
    same_month = forecast_source[forecast_source["date"].dt.month == future_date.month]
    if same_month.empty:
        same_month = forecast_source
    baseline = same_month.iloc[-1]
    row = {
        "year_month": future_date.strftime("%Y-%m"),
        "forecast_method": "seasonal latest-calendar-month baseline",
        "aq_resilience_subscore": float(baseline["aq_resilience_subscore"]),
        "heat_resilience_subscore": float(baseline["heat_resilience_subscore"]),
        "flood_resilience_subscore": float(baseline["flood_resilience_subscore"]),
    }
    row["composite_climate_resilience_score"] = round(
        0.40 * row["aq_resilience_subscore"]
        + 0.35 * row["heat_resilience_subscore"]
        + 0.25 * row["flood_resilience_subscore"],
        1,
    )
    row["resilience_category"] = classify_resilience(row["composite_climate_resilience_score"])
    row["recommended_urban_intervention"] = recommend_intervention(pd.Series(row))
    forecast_rows.append(row)

future_risk_forecast = pd.DataFrame(forecast_rows)
future_risk_forecast.to_csv(PROJECT_ROOT / "Nigdi_Future_Risk_Baseline_Forecast.csv", index=False)
future_risk_forecast

,year_month,forecast_method,aq_resilience_subscore,heat_resilience_subscore,flood_resilience_subscore,composite_climate_resilience_score,resilience_category,recommended_urban_intervention
0,2026-10,seasonal latest-calendar-month baseline,71.5,98.1,66.1,79.5,"High Resilience (Clean, Cool & Safe)","NORMAL OPERATIONS: Continue canopy, drainage, ..."
1,2026-11,seasonal latest-calendar-month baseline,51.6,99.9,81.2,75.9,"High Resilience (Clean, Cool & Safe)","NORMAL OPERATIONS: Continue canopy, drainage, ..."
2,2026-12,seasonal latest-calendar-month baseline,50.5,100.0,88.0,77.2,"High Resilience (Clean, Cool & Safe)","NORMAL OPERATIONS: Continue canopy, drainage, ..."
3,2027-01,seasonal latest-calendar-month baseline,56.0,100.0,89.7,79.8,"High Resilience (Clean, Cool & Safe)","NORMAL OPERATIONS: Continue canopy, drainage, ..."
4,2027-02,seasonal latest-calendar-month baseline,43.0,80.5,90.8,68.1,Moderate Resilience (Satisfactory),AIR POLLUTION ALERT: Activate the correspondin...
5,2027-03,seasonal latest-calendar-month baseline,68.4,54.3,90.8,69.1,Moderate Resilience (Satisfactory),"NORMAL OPERATIONS: Continue canopy, drainage, ..."


In [9]:
artifact_checks = {
    "climate_map": (PROJECT_ROOT / "nigdi_climate_vulnerability_map.html").exists(),
    "dashboard_scoreboard": (PROJECT_ROOT / "Nigdi_TriHazard_Monthly_Master_Scoreboard.csv").exists(),
    "aq_model": (PROJECT_ROOT / "nigdi_next_hour_pm25_gradient_boosting.pkl").exists(),
    "heat_model": (PROJECT_ROOT / "nigdi_next_hour_heat_gradient_boosting.pkl").exists(),
    "future_baseline_forecast": (PROJECT_ROOT / "Nigdi_Future_Risk_Baseline_Forecast.csv").exists(),
}

completion_percent = {
    "climate_vulnerability_maps": 100,
    "adaptation_recommendations": 70,
    "urban_planner_dashboard": 85,
    "future_climate_risk_prediction": 55,
    "independent_resilience_scores": 90,
    "tri_hazard_score_integration": 90,
}
completion_percent["overall_expected_outcomes"] = round(
    sum(completion_percent[key] for key in [
        "climate_vulnerability_maps",
        "adaptation_recommendations",
        "urban_planner_dashboard",
        "future_climate_risk_prediction",
    ]) / 4,
    1,
)

print("=== ARTIFACT CHECKS ===")
for name, exists in artifact_checks.items():
    print(f"{name}: {'PASS' if exists else 'MISSING'}")
print("\n=== COMPLETION PERCENTAGES ===")
for name, value in completion_percent.items():
    print(f"{name}: {value}%")

=== ARTIFACT CHECKS ===
climate_map: PASS
dashboard_scoreboard: PASS
aq_model: PASS
heat_model: PASS
future_baseline_forecast: PASS

=== COMPLETION PERCENTAGES ===
climate_vulnerability_maps: 100%
adaptation_recommendations: 70%
urban_planner_dashboard: 85%
future_climate_risk_prediction: 55%
independent_resilience_scores: 90%
tri_hazard_score_integration: 90%
overall_expected_outcomes: 77.5%
